In [1]:
import os
import json
import hashlib
from abc import ABC, abstractmethod
from pathlib import Path
from datetime import datetime
from enum import Enum
from typing import Union, Iterable, Optional

import numpy as np
import pandas as pd
import yfinance as yf
import statsmodels.api as sm
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.colors as mcolors
import matplotlib.gridspec as gridspec
from scipy import stats

In [2]:
# --- 1. The Abstract Contract ---
class DataAdapter(ABC):
    """
    Abstract Base Class defining the standard interface for all data sources.
    Includes shared caching logic.
    """
    
    def __init__(self, cache_dir: str):
        self.cache_dir = Path(cache_dir)
        if not self.cache_dir.exists():
            os.makedirs(self.cache_dir)

    @abstractmethod
    def get_data(self, tickers: Union[str, Iterable[str], Enum, Iterable[Enum]],
                 start_date: str,
                 end_date: Optional[str] = None) -> pd.DataFrame:
        """
        Fetches data for the given tickers within the date range.
        Args:
            tickers: Single identifier or list of identifiers (str or Enum).
            start_date (str): YYYY-MM-DD start date.
            end_date (str, optional): YYYY-MM-DD end date. Defaults to current date.

        Returns:
            pd.DataFrame: DataFrame with DateTime index.
        """
        pass

    def _get_current_date(self) -> str:
        """Helper to get current date as string."""
        return datetime.today().strftime('%Y-%m-%d')

    def _generate_cache_key(self, tickers: Union[str, Iterable[str], Enum, Iterable[Enum]],
                            start_date: str, end_date: str) -> str:
        """
        Creates a deterministic hash based on tickers and date range.
        Handles both strings and Enums.
        """
        # Normalize tickers to a list
        if isinstance(tickers, (str, Enum)):
            ticker_list = [tickers]
        else:
            ticker_list = list(tickers)

        # Convert Enums to values or strings, ensure list of strings for hashing
        normalized_tickers = []
        for t in ticker_list:
            if isinstance(t, Enum):
                normalized_tickers.append(str(t.value))
            else:
                normalized_tickers.append(str(t))

        # Sort to ensure order doesn't affect hash
        normalized_tickers.sort()

        payload = {
            "tickers": normalized_tickers,
            "start": start_date,
            "end": end_date
        }
        # Serialize to JSON string ensuring deterministic order
        payload_str = json.dumps(payload, sort_keys=True)
        return hashlib.md5(payload_str.encode('utf-8')).hexdigest()

    def _get_cache_path(self, cache_key: str) -> Path:
        """Returns the full path for a cache file."""
        return self.cache_dir / f"{cache_key}.csv"

    def clear_cache(self):
        """Removes all CSV files in the cache directory."""
        for file in self.cache_dir.glob("*.csv"):
            file.unlink()



class YFinanceAdapter(DataAdapter):
    """
    Adapter for Yahoo Finance data.
    Uses shared caching logic from DataAdapter.

    Caching behaviour
    -----------------
    * Same tickers + same date range  →  cache hit, reads from CSV, no network call.
    * Any other combination          →  cache miss, fetches from Yahoo Finance,
                                        writes a *new* CSV (previous caches untouched).
    * end_date > today               →  raises ValueError before any fetch or cache
    """

    def __init__(self, cache_dir: str = "data/yfinance_cache"):
        super().__init__(cache_dir)

    def get_data(self, tickers: Union[str, Iterable[str]],
              start_date: str,
              end_date: Optional[str] = None,
              force_refresh: bool = False) -> pd.DataFrame:
        
        #1) Resolve end_date
        if end_date is None:
            end_date = self._get_current_date()
        
        # ── 2. Guard: future date ────────────────────────────────────────
        today = self._get_current_date()
        if end_date > today:
            raise ValueError(
                f"end_date '{end_date}' is in the future (today is {today}). "
                "No data will be returned."
            )
        if start_date > today:
            raise ValueError(
                f"start_date '{start_date}' is in the future (today is {today}). "
                "No data will be returned."
            )
    
        # ── 3. Normalise tickers → sorted list of uppercase strings ──────
        normalised = self._normalise_tickers(tickers)

        # ── 4. Cache look-up ─────────────────────────────────────────────
        cache_key  = self._generate_cache_key(normalised, start_date, end_date)
        cache_path = self._get_cache_path(cache_key)

        if cache_path.exists() and not force_refresh:
            print(f"[YFinanceAdapter] Cache hit  → {cache_path.name}")
            return self._read_cache(cache_path)

        # ── 5. Remote fetch ──────────────────────────────────────────────
        print(f"[YFinanceAdapter] Cache miss → fetching {normalised} "
              f"from {start_date} to {end_date}")
        df = self._fetch_from_yfinance(normalised, start_date, end_date)

        # ── 6. Write NEW cache file (existing files untouched) ───────────
        self._write_cache(df, cache_path)
        print(f"[YFinanceAdapter] Cached     → {cache_path.name}")

        return df
    
    # ------------------------------------------------------------------
    # Private helpers
    # ------------------------------------------------------------------
    @staticmethod
    def _fetch_from_yfinance(
        tickers: list[str], start_date: str, end_date: str
    ) -> pd.DataFrame:
        """
        Download *Adj Close* prices via yfinance.

        yfinance's end param is *exclusive*, so we add one day so that
        end_date itself is included in the results.
        """
        # Make end_date inclusive
        end_dt_exclusive = (
            datetime.strptime(end_date, "%Y-%m-%d")
            + pd.Timedelta(days=1)
        ).strftime("%Y-%m-%d")

        raw = yf.download(
                tickers=tickers,
                start=start_date,
                end=end_dt_exclusive,
                auto_adjust=True,   # gives clean 'Close' (adjusted)
                progress=False,
        )

        if raw.empty:
            raise ValueError(
                f"yfinance returned no data for {tickers} "
                f"between {start_date} and {end_date}."
            )

        # ── Extract close prices ─────────────────────────────────────────
        # Multi-ticker downloads have a MultiIndex; single-ticker does not.
        if isinstance(raw.columns, pd.MultiIndex):
            df = raw["Close"]
        else:
            # Single ticker: raw has columns like ['Open','High',…,'Close']
            df = raw[["Close"]].rename(columns={"Close": tickers[0]})

        df.index = pd.to_datetime(df.index)
        df.index.name = "Date"
        df.sort_index(inplace=True)
        return df
    
    @staticmethod
    def _normalise_tickers(
        tickers: Union[str, Iterable[str], Enum, Iterable[Enum]]
    ) -> list[str]:
        """Convert any supported ticker representation to a sorted list of
        uppercase strings."""
        if isinstance(tickers, (str, Enum)):
            raw = [tickers]
        else:
            raw = list(tickers)

        result = []
        for t in raw:
            if isinstance(t, Enum):
                result.append(str(t.value).upper())
            else:
                result.append(str(t).upper())

        return sorted(set(result))  # deduplicate + sort for cache-key stability
    
    @staticmethod
    def _read_cache(path: Path) -> pd.DataFrame:
        df = pd.read_csv(path, index_col=0, parse_dates=True)
        df.index.name = "Date"
        return df

    @staticmethod
    def _write_cache(df: pd.DataFrame, path: Path) -> None:
        df.to_csv(path)

In [3]:
#Simple Test of the Yahoo Finance Adaptor is working as intended
class ETF(Enum):
    SPY = "SPY"
    QQQ = "QQQ"

adapter = YFinanceAdapter()

# First call  → remote fetch + writes cache
df1 = adapter.get_data(ETF.SPY, "2019-01-01", "2024-12-31")
df2 = adapter.get_data(ETF.SPY, "2025-01-01", "2025-12-31")

[YFinanceAdapter] Cache hit  → 7f7fa7a8b640460eeb5c401f208baa10.csv
[YFinanceAdapter] Cache hit  → 5244f8e2ccaf6e2697f5af3763af43b2.csv


In [4]:
df1.rename(columns={'SPY': 'Adj Close'}, inplace=True)
df1

,Adj Close
Date,
2019-01-02,224.382523
2019-01-03,219.028183
2019-01-04,226.364655
2019-01-07,228.149445
2019-01-08,230.292969
...,...
2024-12-24,592.702148
2024-12-26,592.741577
2024-12-27,586.502014


In [8]:
df2.rename(columns={'SPY': 'Adj Close'}, inplace=True)
df2

,Adj Close
Date,
2025-01-02,576.280334
2025-01-03,583.485779
2025-01-06,586.847046
2025-01-07,580.213257
2025-01-08,581.060913
...,...
2025-12-24,688.499695
2025-12-26,688.429871
2025-12-29,685.976562


In [9]:
df1.to_csv('./data/spy_train_2019-01-01_2024-12-31.csv')

In [10]:
df2.to_csv('./data/spy_test_2025-01-01_2025-12-31.csv')